In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.decomposition import LatentDirichletAllocation

LSI

In [ ]:
# 1. 데이터 로드 및 확인
dataset = fetch_20newsgroups(shuffle=True, random_state=1, remove=('headers', 'footers', 'quotes'))
documents = dataset.data
print(f"Total documents: {len(documents)}")

# 2. 텍스트 전처리 (데이터프레임 생성 및 특수문자 제거)
news_df = pd.DataFrame({'document': documents})
# 특수 문자 제거
news_df['clean_doc'] = news_df['document'].str.replace("[^a-zA-Z]", " ", regex=True)
# 길이가 3 이하인 단어 제거
news_df['clean_doc'] = news_df['clean_doc'].apply(lambda x: ' '.join([w for w in x.split() if len(w) > 3]))
# 전체 단어 소문자 변환
news_df['clean_doc'] = news_df['clean_doc'].apply(lambda x: x.lower())

# 3. 불용어 제거 및 토큰화
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = stopwords.words('english')

tokenized_doc = news_df['clean_doc'].apply(lambda x: x.split()) # 토큰화
tokenized_doc = tokenized_doc.apply(lambda x: [item for item in x if item not in stop_words]) # 불용어 제거

# 4. TF-IDF 행렬 생성
# 다시 역토큰화하여 벡터화 진행
detokenized_doc = []
for i in range(len(news_df)):
    t = ' '.join(tokenized_doc[i])
    detokenized_doc.append(t)

news_df['detokenized_doc'] = detokenized_doc

vectorizer = TfidfVectorizer(stop_words='english', max_features=1000, max_df=0.5, smooth_idf=True)
X = vectorizer.fit_transform(news_df['detokenized_doc'])
print(f"TF-IDF shape: {X.shape}")

# 5. LSI 모델링 (Truncated SVD)
svd_model = TruncatedSVD(n_components=20, algorithm='randomized', n_iter=100, random_state=122)
svd_model.fit(X)

# 6. 각 토픽별 주요 단어 출력 함수
terms = vectorizer.get_feature_names_out()

def get_topics(components, feature_names, n=5):
    for idx, topic in enumerate(components):
        print("Topic %d:" % (idx+1), [(feature_names[i], topic[i].round(5)) for i in topic.argsort()[:-n - 1:-1]])

print("\n--- LSI 주제 추출 결과 ---")
get_topics(svd_model.components_, terms)

Total documents: 11314


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


TF-IDF shape: (11314, 1000)

--- LSI 주제 추출 결과 ---
Topic 1: [('like', np.float64(0.2138)), ('know', np.float64(0.20031)), ('people', np.float64(0.19334)), ('think', np.float64(0.17802)), ('good', np.float64(0.15105))]
Topic 2: [('thanks', np.float64(0.32918)), ('windows', np.float64(0.29093)), ('card', np.float64(0.18016)), ('drive', np.float64(0.1739)), ('mail', np.float64(0.15131))]
Topic 3: [('game', np.float64(0.37159)), ('team', np.float64(0.32533)), ('year', np.float64(0.28205)), ('games', np.float64(0.25416)), ('season', np.float64(0.18464))]
Topic 4: [('drive', np.float64(0.52823)), ('scsi', np.float64(0.20043)), ('disk', np.float64(0.15518)), ('hard', np.float64(0.15511)), ('card', np.float64(0.14049))]
Topic 5: [('windows', np.float64(0.40544)), ('file', np.float64(0.25619)), ('window', np.float64(0.1806)), ('files', np.float64(0.16196)), ('program', np.float64(0.14009))]
Topic 6: [('know', np.float64(0.35192)), ('like', np.float64(0.24617)), ('think', np.float64(0.24285)), ('

LDA

In [ ]:
# 1. 데이터 로드 및 확인 
dataset = fetch_20newsgroups(shuffle=True, random_state=1, remove=('headers', 'footers', 'quotes'))
documents = dataset.data
print(f"Total documents: {len(documents)}")

# 2. 텍스트 전처리 (데이터프레임 생성 및 특수문자 제거) 
news_df = pd.DataFrame({'document': documents})

# 특수 문자 제거
news_df['clean_doc'] = news_df['document'].str.replace("[^a-zA-Z]", " ", regex=True)
# 길이가 3 이하인 단어 제거
news_df['clean_doc'] = news_df['clean_doc'].apply(lambda x: ' '.join([w for w in x.split() if len(w) > 3]))
# 전체 단어 소문자 변환
news_df['clean_doc'] = news_df['clean_doc'].apply(lambda x: x.lower())

# 3. 불용어 제거를 위한 토큰화 
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = stopwords.words('english')

tokenized_doc = news_df['clean_doc'].apply(lambda x: x.split()) # 토큰화
tokenized_doc = tokenized_doc.apply(lambda x: [item for item in x if item not in stop_words]) # 불용어 제거

# 4. TF-IDF 행렬 생성 (LDA 모델의 입력으로 사용) 
# 역토큰화 (Detokenization)
detokenized_doc = []
for i in range(len(news_df)):
    t = ' '.join(tokenized_doc[i])
    detokenized_doc.append(t)

news_df['detokenized_doc'] = detokenized_doc

# 상위 1,000개의 단어를 대상으로 TF-IDF 행렬 생성
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000, max_df=0.5, smooth_idf=True)
X = vectorizer.fit_transform(news_df['detokenized_doc'])
print(f"TF-IDF shape: {X.shape}")

# 5. LDA 모델링 
# 10개의 토픽을 추출하도록 설정
lda_model = LatentDirichletAllocation(n_components=10, learning_method='online', random_state=777, max_iter=1)
lda_top = lda_model.fit_transform(X)

# 6. 각 토픽별 주요 단어 출력 함수 
terms = vectorizer.get_feature_names_out()

def get_topics(components, feature_names, n=5):
    for idx, topic in enumerate(components):
        print("Topic %d:" % (idx+1), [(feature_names[i], topic[i].round(5)) for i in topic.argsort()[:-n - 1:-1]])

print("\n--- LDA 주제 추출 결과 ---")
get_topics(lda_model.components_, terms)

Total documents: 11314


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


TF-IDF shape: (11314, 1000)

--- LDA 주제 추출 결과 ---
Topic 1: [('people', np.float64(236.53626)), ('think', np.float64(180.60114)), ('like', np.float64(171.57708)), ('know', np.float64(156.57161)), ('time', np.float64(146.78101))]
Topic 2: [('space', np.float64(45.63616)), ('nasa', np.float64(23.98544)), ('orbit', np.float64(20.27583)), ('satellite', np.float64(20.02513)), ('moon', np.float64(15.81806))]
Topic 3: [('thanks', np.float64(164.02061)), ('windows', np.float64(136.63212)), ('know', np.float64(122.39339)), ('card', np.float64(106.11724)), ('like', np.float64(106.01977))]
Topic 4: [('bike', np.float64(69.7014)), ('engine', np.float64(29.907)), ('miles', np.float64(26.92137)), ('like', np.float64(23.39827)), ('ground', np.float64(21.09653))]
Topic 5: [('chip', np.float64(43.45536)), ('clipper', np.float64(34.63778)), ('algorithm', np.float64(30.13795)), ('keys', np.float64(27.04699)), ('encryption', np.float64(25.76183))]
Topic 6: [('game', np.float64(90.60337)), ('games', np.floa